In [ ]:
# IMPORTANT: SOME KAGGLE DATA SOURCES ARE PRIVATE
# RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES.
import kagglehub
kagglehub.login()


In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

fleet_partnership_prediction_path = kagglehub.competition_download('fleet-partnership-prediction')

print('Data source import complete.')


# Fleet Partnership Prediction - Competition Notebook

## Competition Overview

This notebook provides a **baseline solution** for the Fleet Partnership Prediction Challenge.

**Goal**: Predict whether a prospective company will become a partner (1) or not (0) based on 100 features.

**Evaluation Metric**: AUC-ROC (Area Under the Receiver Operating Characteristic Curve)

**Target**: Highest AUC to support your company's decision!

---

This notebook demonstrates a ML pipeline including:
- Data loading and exploration
- Missing value handling
- Outlier detection and removal
- Feature normalization
- Multiple modeling approaches
- Cross-validation for model selection
- Submission file generation

**Course**: Part of كورس البداية www.aibdaya.com AI & Data Science course by Dr. Mahmoud Eid  
**Website**: www.aibdaya.com

In [ ]:
!nvidia-smi

Fri Jan 23 15:16:48 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 570.172.08             Driver Version: 570.172.08     CUDA Version: 12.8     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   39C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

## 1. Import Libraries

In [ ]:
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.feature_selection import SequentialFeatureSelector
from sklearn.model_selection import KFold, cross_val_score
from sklearn.metrics import roc_auc_score, confusion_matrix, classification_report
import warnings
warnings.filterwarnings('ignore')

# Set random seeds for reproducibility
np.random.seed(42)
torch.manual_seed(42)

print("Libraries imported successfully")

Libraries imported successfully


## 2. Configuration

## 3. Load Data

Load the training and testing datasets:
- Training data: 100 features + target variable
- Testing data: 100 features only (no target - this is what your model should predict)

In [ ]:
# Deep Neural Network Configuration
EPOCHS = 100
BATCH_SIZE = 64
LEARNING_RATE = 0.001


TRAIN_PATH = '/kaggle/input/fleet-partnership-prediction/train.csv'
TEST_PATH = '/kaggle/input/fleet-partnership-prediction/test.csv'

print(f"Configuration:")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch Size: {BATCH_SIZE}")
print(f"  Learning Rate: {LEARNING_RATE}")
print(f"  Training Data: {TRAIN_PATH}")
print(f"  Testing Data: {TEST_PATH}")

Configuration:
  Epochs: 100
  Batch Size: 64
  Learning Rate: 0.001
  Training Data: /kaggle/input/fleet-partnership-prediction/train.csv
  Testing Data: /kaggle/input/fleet-partnership-prediction/test.csv


In [ ]:
def load_data(train_path=TRAIN_PATH, test_path=TEST_PATH):
    """
    Load training and testing data from CSV files.

    Returns:
        X_train, y_train, X_test: Feature matrices and training target
    """
    print("\n" + "="*50)
    print("LOADING DATA")
    print("="*50)

    train_df = pd.read_csv(train_path)
    test_df = pd.read_csv(test_path)

    # Separate features and target for training data
    X_train = train_df.drop('target', axis=1).values
    y_train = train_df['target'].values

    # Test data - drop 'id' column if present, and 'target' if present
    cols_to_drop = [col for col in ['id', 'target'] if col in test_df.columns]        # The ID number If you include it, your model might try to learn patterns from random numbers, which is meaningless and can hurt performance.
    if cols_to_drop:
        X_test = test_df.drop(cols_to_drop, axis=1).values
    else:
        X_test = test_df.values

    print(f"Training data: {X_train.shape[0]} samples, {X_train.shape[1]} features")
    print(f"Test data: {X_test.shape[0]} samples, {X_test.shape[1]} features")
    print(f"Missing values in training: {np.sum(np.isnan(X_train))}")
    print(f"Class distribution in training: {np.bincount(y_train)}")

    return X_train, y_train, X_test

# Load the data
X_train, y_train, X_test = load_data()


LOADING DATA
Training data: 38400 samples, 100 features
Test data: 9600 samples, 100 features
Missing values in training: 38399
Class distribution in training: [20263 18137]


In [ ]:
# test_df = pd.read_csv(TEST_PATH)
# test_df.columns
# # cols_to_drop = [col for col in ['id', 'target'] if col in test_df.columns]
# # cols_to_drop

## 4. Handle Missing Values

We use **median imputation** to handle missing values:
- More robust to outliers than mean imputation
- Calculate imputation values from training data only
- Apply the same values to test data (no data leakage)

In [ ]:
def handle_missing_values(X_train, X_test, strategy='median'):
    """
    Impute missing values using training statistics.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix
        strategy: 'median' or 'mean'

    Returns:
        X_train_imputed, X_test_imputed: Imputed feature matrices
    """
    print("\n" + "="*50)
    print("HANDLING MISSING VALUES")
    print("="*50)

    X_train_imputed = X_train.copy()
    X_test_imputed = X_test.copy()

    if strategy == 'median':
        impute_values = np.nanmedian(X_train, axis=0)
        print(f"Using median imputation (robust to outliers)")
    else:
        impute_values = np.nanmean(X_train, axis=0)
        print(f"Using mean imputation")

    for feature_idx in range(X_train.shape[1]):
        missing_mask_train = np.isnan(X_train_imputed[:, feature_idx])
        X_train_imputed[missing_mask_train, feature_idx] = impute_values[feature_idx]

        # Test set (using training statistics)
        missing_mask_test = np.isnan(X_test_imputed[:, feature_idx])
        X_test_imputed[missing_mask_test, feature_idx] = impute_values[feature_idx]

    print(f"Missing values after imputation:")
    print(f"  Training: {np.sum(np.isnan(X_train_imputed))}")
    print(f"  Test: {np.sum(np.isnan(X_test_imputed))}")

    return X_train_imputed, X_test_imputed

X_train, X_test = handle_missing_values(X_train, X_test, strategy='median')


HANDLING MISSING VALUES
Using median imputation (robust to outliers)
Missing values after imputation:
  Training: 0
  Test: 0


## 5. Detect and Remove Outliers

We use the **IQR (Interquartile Range) method** to detect outliers:
- Calculate Q1 (25th percentile) and Q3 (75th percentile)
- IQR = Q3 - Q1
- Outliers are values < Q1 - 1.5*IQR or > Q3 + 1.5*IQR


In [ ]:
import numpy as np

def cap_outliers_percentile(X_train, X_test, lower_percentile=5, upper_percentile=95):
    """
    Caps outliers in both Train and Test sets using statistics derived
    ONLY from the Training set to prevent data leakage.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix
        lower_percentile: Lower bound percentile (default 5)
        upper_percentile: Upper bound percentile (default 95)

    Returns:
        X_train_capped, X_test_capped: Transformed feature matrices
    """
    print("\n" + "="*50)
    print(f"CAPPING OUTLIERS (Winsorization)")
    print(f"Bounds derived from Train: {lower_percentile}th - {upper_percentile}th percentiles")
    print("="*50)

    X_train_capped = X_train.copy()
    X_test_capped = X_test.copy()
    n_features = X_train.shape[1]

    total_capped_train = 0
    total_capped_test = 0

    for feature_idx in range(n_features):
        train_feature_data = X_train[:, feature_idx]

        lower_bound = np.percentile(train_feature_data, lower_percentile)  # get index
        upper_bound = np.percentile(train_feature_data, upper_percentile)

        outliers_train = (X_train_capped[:, feature_idx] < lower_bound) | \
                         (X_train_capped[:, feature_idx] > upper_bound)      # False, True
        total_capped_train += np.sum(outliers_train)

        X_train_capped[:, feature_idx] = np.clip(
            X_train_capped[:, feature_idx], lower_bound, upper_bound
        )

        outliers_test = (X_test_capped[:, feature_idx] < lower_bound) | \
                        (X_test_capped[:, feature_idx] > upper_bound)
        total_capped_test += np.sum(outliers_test)

        X_test_capped[:, feature_idx] = np.clip(
            X_test_capped[:, feature_idx], lower_bound, upper_bound
        )

    print(f"{(total_capped_train / (X_train.shape[0] * n_features)) * 100} %")
    print(f"{(total_capped_test / (X_test.shape[0] * X_test.shape[1])) * 100:.2f} %")
    return X_train_capped, X_test_capped

# Apply capping/winsorization
X_train, X_test = cap_outliers_percentile(X_train, X_test, lower_percentile=5, upper_percentile=95)

print(f"Shape of training: {X_train.shape}")
print(f"Shape of testing: {X_test.shape}")


CAPPING OUTLIERS (Winsorization)
Bounds derived from Train: 5th - 95th percentiles
10.0 %
10.08 %
Shape of training: (38400, 100)
Shape of testing: (9600, 100)


| Outlier Percentage | Action | Reasoning |
| --- | --- | --- |
| **< 5%** | ✅ **Safe to DELETE** | Minimal data loss, clean dataset |
| **5-10%** | ⚠️ **CAP (recommended)** | Borderline - capping is safer |
| **> 10%** | ❌ **MUST CAP** | Too much data loss if deleted |

Your data: 10% outliers per feature
→ Use CAPPING ✅ (your current approach)

In [ ]:
# Calc percentile manually
# x_len = len(X_train[:, 0])-1
# x_s = np.sort(X_train[:, 0])
# ind = 0.9*x_len
# deff = ind - np.floor(ind)  #0.09
# v_f = x_s[int(np.floor(ind))]  # at 8 --> 3.596397211014225
# v_c = x_s[int(np.ceil(ind))]  # at 9 --> 3.597906040567908
# v_f_a = deff*(v_c-v_f)
# r = v_f+v_f_a
# r
# np.max(X_train[:,1])

In [ ]:
# X_train_capped = X_train.copy()
# train_feature_data = X_train[:, 1]
# lower_bound = np.percentile(train_feature_data, 5)
# upper_bound = np.percentile(train_feature_data, 95)
# sum((X_train_capped[:, 1] < lower_bound) | (X_train_capped[:, 1] > upper_bound))
# X_train_capped[:, 1] = np.clip(
#             X_train_capped[:, 1], lower_bound, upper_bound
#         )
# pd.DataFrame(np.sort(X_train_capped[:, 1]))

## 6. Check Feature Correlations

Check for highly correlated features.

In [ ]:
def check_correlation(X, threshold=0.8):
    """
    Check for highly correlated features.

    Args:
        X: Feature matrix
        threshold: Correlation threshold

    Returns:
        corr_matrix: Correlation matrix
    """
    print("\n" + "="*50)
    print("CHECKING FEATURE CORRELATIONS")
    print("="*50)

    # Calculate correlation matrix
    corr_matrix = np.corrcoef(X.T)

    # Find high correlations (excluding diagonal)
    high_corr_pairs = []
    n_features = X.shape[1]

    for i in range(n_features):
        for j in range(i+1, n_features):
            if abs(corr_matrix[i, j]) > threshold:
                high_corr_pairs.append((i, j, corr_matrix[i, j]))

    if len(high_corr_pairs) == 0:
        print(f"No high correlations found (threshold={threshold})")
        print(f"No significant multicollinearity detected")
    else:
        print(f"Found {len(high_corr_pairs)} feature pairs with correlation > {threshold}")
        for i, j, corr in high_corr_pairs[:5]:  # Show first 5
            print(f"  - Feature {i} & Feature {j}: {corr:.3f}")

    return corr_matrix

# Check correlations
corr_matrix = check_correlation(X_train, threshold=0.8)


CHECKING FEATURE CORRELATIONS
No high correlations found (threshold=0.8)
No significant multicollinearity detected


## 7. Feature Normalization (Z-Score)

Apply **Z-score normalization** to standardize features
Calculate statistics from training data only to avoid data leakage

In [ ]:
def apply_zscore(X_train, X_test):
    """
    Apply Z-score normalization.

    Args:
        X_train: Training feature matrix
        X_test: Test feature matrix

    Returns:
        X_train_scaled, X_test_scaled: Normalized feature matrices
    """
    print("\n" + "="*50)
    print("APPLYING Z-SCORE NORMALIZATION")
    print("="*50)

    mean = np.mean(X_train, axis=0)
    std = np.std(X_train, axis=0)
    # Avoid division by zero
    std[std == 0] = 1

    # Apply to both train and test
    X_train_scaled = (X_train - mean) / std
    X_test_scaled = (X_test - mean) / std

    print(f"Normalized {X_train.shape[1]} features")
    print(f"Training data: mean={np.mean(X_train_scaled):.6f}, std={np.std(X_train_scaled):.6f}")
    print(f"Test data: mean={np.mean(X_test_scaled):.6f}, std={np.std(X_test_scaled):.6f}")

    return X_train_scaled, X_test_scaled

X_train_scaled, X_test_scaled = apply_zscore(X_train, X_test)


APPLYING Z-SCORE NORMALIZATION
Normalized 100 features
Training data: mean=-0.000000, std=1.000000
Test data: mean=-0.000302, std=1.001269


## 8. Model Training and Evaluation Functions

Define helper functions for model training and evaluation using cross-validation.

In [ ]:
def evaluate_model_cv(model, X, y, model_type='sklearn'):
    """
    Evaluate model on training data using AUC-ROC.

    Args:
        model: Trained model
        X: Feature matrix
        y: Target vector
        model_type: 'sklearn' or 'pytorch'

    Returns:
        auc_score: AUC-ROC score on training data
    """
    if model_type == 'sklearn':
        y_pred_proba = model.predict_proba(X)[:, 1]
    else:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X)
            y_pred_proba = model(X_tensor).numpy().flatten()

    auc_score = roc_auc_score(y, y_pred_proba)
    return auc_score

## 9. Experiment 1: Logistic Regression (Baseline)



✅ **First one (train_logistic_regression)**:
- Trains model on ALL training data
- This is the final model you use for predictions on test data

✅ **Second one (cross_validate_model)**:
- Trains 5 different models (one per fold)
- Each model trains on 80% and validates on 20%
- Gives you expected performance scores
- These 5 models are **thrown away** after getting the scores

---
they're doing cross-validation FIRST to **estimate** performance, THEN training the final model.
What happens inside cross_validate_model:
```python
def cross_validate_model(X_train, y_train, ...):
    # Define model configuration (NOT trained yet!)
    model = LogisticRegression(penalty=penalty, C=C, ...)
    
    # cross_val_score does this internally:
    # Fold 1: Train on 80%, test on 20% → Score 1
    # Fold 2: Train on 80%, test on 20% → Score 2
    # Fold 3: Train on 80%, test on 20% → Score 3
    # Fold 4: Train on 80%, test on 20% → Score 4
    # Fold 5: Train on 80%, test on 20% → Score 5
    
    cv_scores = cross_val_score(model, X_train, y_train, cv=5)
    # Returns: [0.84, 0.86, 0.85, 0.85, 0.83]
    
    return np.mean(cv_scores), np.std(cv_scores), cv_scores
```
1. **cross_validate_model**:
    1. Takes the model **configuration** (not trained)
    2. Internally trains it 5 times on different data splits
    3. Returns 5 scores
    4. **Throws away all 5 trained models**

2. **train_logistic_regression**:
   - "Now train once on ALL data and give me the final model"
   - Returns: Actual model object you can use
   - `predictions = model.predict(X_test)` ✅

---
**penalty=penalty** → Regularization Type ✅
```python
penalty='l2'  # Ridge regularization (shrink all weights)
penalty='l1'  # Lasso regularization (can set some weights to 0)
```
**What regularization does:**
Prevents the model from relying too heavily on any single feature by penalizing large weights.

---
2. **C=C** → Inverse Regularization Strength
This controls **how much regularization** to apply.
Important: It's INVERSE!
- **Large C** (e.g., C=10): **LESS** regularization → more freedom → more complex model
- **Small C** (e.g., C=0.1): **MORE** regularization → less freedom → simpler model
**Think of C as "freedom level":**
```python
C = 0.01  → Very strict, heavy regularization
C = 1.0   → Moderate regularization (default)
C = 100   → Very loose, almost no regularization
```
**Mathematical relationship:**
Regularization strength = 1/C
C = 10  → strength = 1/10 = 0.1 (weak regularization)
C = 0.1 → strength = 1/0.1 = 10 (strong regularization)

---
**solver='liblinear'** → Optimization Algorithm
This is the **algorithm used to find the best weights**.
Think of it like different methods to climb a mountain (find the optimal solution):
- `'liblinear'`: Good for small/medium datasets, supports L1 and L2
- `'lbfgs'`: Good for large datasets, only supports L2
- `'saga'`: Good for very large datasets, supports L1 and L2

---
**max_iter=1000** → Maximum Iterations (NOT epochs!)
**For Logistic Regression:**
- `max_iter` = maximum number of **optimization steps**
- This is NOT the same as epochs in neural networks!
- `max_iter` is a **limit** (might stop early), while `epochs` usually runs completely.

Logistic Regression uses an optimization algorithm that:
Starts with random weights --> Calculates the error --> Adjusts weights to reduce error --> Repeats until:
   - ✅ Converges (error stops improving) → might stop at iteration 150
   - ⏱️ Reaches max_iter=1000 → forced stop

**Example:**
```python
max_iter=1000
# Training might look like:
Iteration 1:   Error = 0.500
Iteration 100: Error = 0.210
Iteration 150: Error = 0.209
Iteration 151: Error = 0.209  ← Converged! Stop here, because No change! (no improvement!) --OR -- ✅ Change is less than tolerance threshold (very tiny)
# Never reaches iteration 1000 because it converged early
```
1. **Each iteration** looks at **ALL samples at once** (or batches, depending on solver)
2. Calculates the **total error** across all samples
3. Updates the weights to reduce that total error
4. Repeats
```python
max_iter = 1000
# Iteration 1: Look at all 1000 samples --> Calculate total error = 0.500  --> Update weights to reduce error
# Iteration 2: Look at all 1000 samples again (with new weights)  -->  Calculate total error = 0.450 (improved!)  -->  Update weights again
# ... continues ...
The goal is to reach the **global minimum** (lowest possible error).
Error
  |
  |     Start here (random weights)
  |        ↓
0.5|       ●
  |        \
0.4|         \
  |          \
0.3|           \
  |            ●  ← Iteration 50
0.2|             \___●___●___●  ← Iterations 100-151 (converged!)
  |                  Global minimum
  +--------------------------------→ Iterations
```
**max_iter IS like epochs!**
**Both pass through all the data each time:**
**Neural Networks:**
```python
epochs = 100
# Epoch 1: Pass through all data, update weights
# Epoch 2: Pass through all data, update weights
# ...
```
**Logistic Regression:**
```python
max_iter = 1000
# Iteration 1: Pass through all data, update weights
# Iteration 2: Pass through all data, update weights
# ...
```
**So what's the difference then?**
- Deep Learning: calls it "**epochs**"
- Traditional ML: calls it "**iterations**"
  `Because Early stopping:`
- Logistic Regression: **Usually** stops early when converged (before reaching max_iter)
- Deep Learning: **Usually** runs all epochs (unless you manually add early stopping)
```python
model = NeuralNetwork()
model.fit(X, y, epochs=100)  # Each epoch: see all data once

model = LogisticRegression(max_iter=100)
model.fit(X, y)  # Each iteration: see all data once
```
---
**n_jobs=-1** → Use All CPU Core
**n_jobs** = "number of jobs/tasks to run in parallel"
```python
n_jobs=1   → Use 1 CPU core (sequential, slow)
n_jobs=4   → Use 4 CPU cores (parallel)
n_jobs=-1  → Use ALL available CPU cores (maximum speed)
```
How it speeds up cross-validation:
**Without parallelization (n_jobs=1):**
```
Core 1: [Fold 1] → [Fold 2] → [Fold 3] → [Fold 4] → [Fold 5]
Total time: 5 minutes (1 min per fold)
```
**With parallelization (n_jobs=-1, assume 4 cores):**
```
Core 1: [Fold 1] (1 min)  } All running
Core 3: [Fold 3] (1 min)  } at the
Core 4: [Fold 4] (1 min)  } same time!
Core 5: [Fold 5] (1 min)
Total time: 1 minutes instead of 5!
```

In [ ]:
def train_logistic_regression(X_train, y_train, penalty='l2', C=1.0):
    """
    Train Logistic Regression model.

    Args:
        X_train: Training features
        y_train: Training target
        penalty: Regularization type ('l1' or 'l2')
        NOTE: L2 (Ridge): Shrinks all feature weights, but keeps them all -- L1 (Lasso): Can set some feature weights to exactly 0 (feature selection)
        C: INVERSE regularization strength -> Larger C (like C=10): Less regularization → model can be more complex -- Smaller C (like C=0.1): More regularization → model forced to be simpler

    Returns:
        model: Trained model
    """
    print(f"\nTraining Logistic Regression (penalty={penalty}, C={C})")
    model = LogisticRegression(
        penalty=penalty,
        C=C,
        solver='liblinear',  # optimization algorithm
        max_iter=1000,
        random_state=42
    )
    model.fit(X_train, y_train)

    if penalty == 'l1':
        n_nonzero = np.sum(model.coef_ != 0)    # With L1, some coefficients become exactly 0, meaning those features are ignored. This tells you how many features the model actually used.

        print(f"L1 regularization selected {n_nonzero} non-zero features")

    print(f"Model trained successfully")
    return model

def cross_validate_model(X_train, y_train, penalty='l2', C=1.0, n_folds=5):
    """
    Perform k-fold cross-validation for Logistic Regression.
    """
    model = LogisticRegression(
        penalty=penalty,
        C=C,                  # This controls how much regularization to apply.  -> Large C (e.g., C=10): LESS regularization → more freedom → more complex model,  ----   Small C (e.g., C=0.1): MORE regularization → less freedom → simpler model
        solver='liblinear',
        max_iter=1000,
        random_state=42
    )

    cv_scores = cross_val_score(
        model, X_train, y_train,
        cv=n_folds,
        scoring='roc_auc',
        n_jobs=-1           #  Use all CPU cores to speed up training (trains the 5 folds in parallel at once (if you have 5+ cores))
    )

    return np.mean(cv_scores), np.std(cv_scores), cv_scores

print("\n" + "="*50)
print("EXPERIMENT 1: Logistic Regression (Baseline)")
print("="*50)

# Cross-validation to estimate performance
lr_cv_mean, lr_cv_std, lr_cv_scores = cross_validate_model(X_train_scaled, y_train, penalty='l2', C=1.0, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"Individual folds: {lr_cv_scores}")

# Train final model on all training data
lr_model = train_logistic_regression(X_train_scaled, y_train, penalty='l2', C=1.0)
lr_train_auc = evaluate_model_cv(lr_model, X_train_scaled, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"Logistic Regression Results:")
print(f"  Training AUC-ROC: {lr_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_cv_mean:.4f} +/- {lr_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 1: Logistic Regression (Baseline)

5-Fold CV AUC-ROC: 0.5107 +/- 0.0054
Individual folds: [0.5169529  0.5019617  0.50710292 0.51352062 0.51395509]

Training Logistic Regression (penalty=l2, C=1.0)
Model trained successfully

Logistic Regression Results:
  Training AUC-ROC: 0.5343
  CV AUC-ROC: 0.5107 +/- 0.0054


***Note: we've used linear classifier but the result has been bad, so this indicates that data might be nonlinear***
Let's use PCA and see if it'll be linear or not

## 10. Experiment 2: PCA + Logistic Regression


**PCA (Principal Component Analysis)** = Dimensionality reduction technique
It takes your **many features** and creates **fewer new features** that capture most of the important information.

---
Simple Analogy:
Imagine you're describing a person with 100 measurements:
- Height, weight, arm length, leg length, shoulder width, chest size, waist size, etc.
Many of these are **correlated** (related):
- Tall people usually have long arms
- Heavy people usually have large waist and chest
**PCA says:** "Instead of 100 measurements, I can describe this person with just 5-10 'principal components' that capture most of the variation."
- You went from **100 features → 30 features**
---
This tells you: **"How much information does each component capture?"**
```python
pca.explained_variance_ratio_ = [0.25, 0.15, 0.10, 0.08, 0.07, ...(sum of all 30)]  --> = 850/1000 = 0.85  = 85% of total variance captured!
#                                 PC1   PC2   PC3   PC4   PC5   ...

# Component 1: Captures (250 units) 25% of total variance
# Component 2: Captures (150 units) 15% of total variance
# Component 3: Captures (100 units) 10% of total variance
# ... and so on
```
- **Variance** = How much the data varies/spreads out
- **High variance** = Data points are spread out (lots of information)   e.g. Feature B: [5, 50, 100, 2, 85]         → High variance (lots of info!)
- **Low variance** = Data points are close together (less information)   e.g. Feature A: [10, 10.1, 9.9, 10.2, 9.8]  → Low variance (not much info)
```python
explained_variance = np.sum(pca.explained_variance_ratio_)  # Add up how much variance all 30 components capture
```
**What this means:**
"By reducing from 100 features to 30 features, we kept **85%** of the information!"
**You threw away 15% of information**, but kept the most important 85%!

**Benefits of PCA:**
1. **Faster training** (30 features vs 100 features)
2. **Less overfitting** (fewer features = simpler model)
3. **Removes noise** (keeps important patterns, discards random variation)

**Trade-off:**
- ✅ Simpler, faster model
- ❌ Lost some information (15% in this case)

```python
Explained variance: 50%  → Bad! Lost too much information
Explained variance: 70%  → Okay, but might lose important patterns
Explained variance: 85%  → Good! Kept most information
Explained variance: 95%  → Great! Almost all information retained
Explained variance: 99%  → Excellent, but maybe not much compression
```
**Rule of thumb:** Aim for **80-95%** explained variance

```
Original data (100 dimensions):
[Complex cloud in 100D space - hard to visualize!]

After PCA (30 dimensions):
[Compressed cloud that captures main shape/patterns]
    ↑
Kept 85% of the "shape" of the data
```
Each component captures a **portion** of variance. Adding them up gives you the **total** variance captured by all 30 components together.
```
Feature 2
        ↑
     10 |     ●
        |    ●/ ●      ← PC1 (main direction)
      5 |   ●/   ●       captures most spread
        |  ●/     ●
      0 +──────────→ Feature 1
        0    5   10
PC2 (second component) captures the remaining vertical/horizontal spread = 15%
If you keep only PC1, you captured 85% of the pattern but lost 15% of detail.
```
![image.png](attachment:524a411f-3fab-42e2-b6e8-fd17978eafd5.png)


```
If you fit PCA separately on test data, you're using information from the test set that your model shouldn't know about!

Training data cloud:
    Feature 2
        ↑
        |    ●●●
        |  ●●●●●    
        | ●●●●●     Main direction: →
        |●●●●●      
        +────────→ Feature 1

PCA finds: "Main spread is diagonal (PC1 →)"

# Apply SAME transformation to test
# Fit on train only
pca.fit(X_train)
X_test_pca = pca.transform(X_test)  # ✅ GOOD!

Test data cloud:
    Feature 2
        ↑
        | ●
        | ●●
        | ●●●     
        | ●●●●    
        +────────→ Feature 1
        
Apply SAME diagonal direction from training →
Even if test data spreads differently!  ↑
```


In [ ]:
def apply_pca(X_train, X_test, n_components=30):
    """
    Apply PCA dimensionality reduction.
    """
    print(f"\nApplying PCA (n_components={n_components})")

    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train)  # Learn the patterns from training data -> # Apply those patterns to training data
    X_test_pca = pca.transform(X_test)        # Apply THE SAME patterns that we get from traning data to test data

    explained_variance = np.sum(pca.explained_variance_ratio_)

    print(f"Reduced from {X_train.shape[1]} to {n_components} principal components")
    print(f"Explained variance: {explained_variance*100:.2f}%")

    return X_train_pca, X_test_pca, pca

print("\n" + "="*50)
print("EXPERIMENT 2: PCA + Logistic Regression")
print("="*50)
# Reduce Features from 100 to 10 PCs
X_train_pca, X_test_pca, pca = apply_pca(X_train_scaled, X_test_scaled, n_components=30)

# Cross-validation
lr_pca_cv_mean, lr_pca_cv_std, _ = cross_validate_model(X_train_pca, y_train, penalty='l2', C=1.0, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_pca_cv_mean:.4f} +/- {lr_pca_cv_std:.4f}")

# Train final model
lr_pca_model = train_logistic_regression(X_train_pca, y_train, penalty='l2', C=1.0)
lr_pca_train_auc = evaluate_model_cv(lr_pca_model, X_train_pca, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"PCA + Logistic Regression Results:")
print(f"  Training AUC-ROC: {lr_pca_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_pca_cv_mean:.4f} +/- {lr_pca_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 2: PCA + Logistic Regression

Applying PCA (n_components=30)
Reduced from 100 to 30 principal components
Explained variance: 30.83%

5-Fold CV AUC-ROC: 0.5023 +/- 0.0064

Training Logistic Regression (penalty=l2, C=1.0)
Model trained successfully

PCA + Logistic Regression Results:
  Training AUC-ROC: 0.5167
  CV AUC-ROC: 0.5023 +/- 0.0064


![image.png](attachment:51597702-d137-434a-b058-2532687af564.png)
* **30.83% is actually quite LOW!**  "30.83% of the data's variation patterns are captured"
* **Lost 69.17%:** The remaining variance is in the other 70 dimensions you didn't keep.
This means:
- ❌ You're losing **69.17%** of the information
- ❌ Those 30 components don't capture most of the patterns
- ⚠️ Your model performance might suffer

* Your **30.83%** suggests the 100 features have information spread across many dimensions, and 30 components isn't enough to capture it all.

*Thus the info NOT in highest variance direction*

**PCA can't compress well because there's no redundancy to eliminate!**

## 11. Experiment 3: L1 Regularization (Sparse Feature Selection)

Use L1 regularization for embedded feature selection.

**Regularization controls WEIGHT SIZE** --> ***High regularization "C" = Make weights smaller***
```python
# Without regularization:
Feature 1 weight: 5.2
Feature 2 weight: 3.8
Feature 3 weight: 7.1
Feature 4 weight: 2.9

# With HIGH regularization (C=0.1):
Feature 1 weight: 0.8  ← Shrunk!
Feature 2 weight: 0.5  ← Shrunk!
Feature 3 weight: 1.2  ← Shrunk!
Feature 4 weight: 0.3  ← Shrunk!

# All 4 features still exist! Just smaller weights.
```
**L2 Regularization (penalty='l2'):**
- Shrinks all weights
- **NEVER sets weights to exactly 0**
- All features remain active

```python
# L2 with C=0.1 (high regularization)
Feature 1: 0.8  ← Small but NOT zero
Feature 4: 0.3  ← Small but NOT zero
# ALL features still used!
```
**L1 Regularization (penalty='l1'):** ← This is what you're using!
- Shrinks weights
- **CAN set some weights to EXACTLY 0**
- This effectively removes features!
```python
# L1 with C=0.1 (high regularization)
Feature 1: 0.0  ← ZERO! Feature removed!
Feature 2: 0.5  ← Still active
Feature 3: 1.8  ← Still active
Feature 4: 0.0  ← ZERO! Feature removed!
# Only 2 out of 4 features used!
```

In [ ]:
print("\n" + "="*50)
print("EXPERIMENT 3: L1 Regularization + Logistic Regression")
print("="*50)

# Cross-validation
lr_l1_cv_mean, lr_l1_cv_std, _ = cross_validate_model(X_train_scaled, y_train, penalty='l1', C=0.1, n_folds=5)
print(f"\n5-Fold CV AUC-ROC: {lr_l1_cv_mean:.4f} +/- {lr_l1_cv_std:.4f}")

# Train final model
lr_l1_model = train_logistic_regression(X_train_scaled, y_train, penalty='l1', C=0.1)
lr_l1_train_auc = evaluate_model_cv(lr_l1_model, X_train_scaled, y_train, model_type='sklearn')

print(f"\n{'='*50}")
print(f"L1 Regularization Results:")
print(f"  Training AUC-ROC: {lr_l1_train_auc:.4f}")
print(f"  CV AUC-ROC: {lr_l1_cv_mean:.4f} +/- {lr_l1_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 3: L1 Regularization + Logistic Regression

5-Fold CV AUC-ROC: 0.5113 +/- 0.0053

Training Logistic Regression (penalty=l1, C=0.1)
L1 regularization selected 87 non-zero features
Model trained successfully

L1 Regularization Results:
  Training AUC-ROC: 0.5342
  CV AUC-ROC: 0.5113 +/- 0.0053


## 12. Experiment 4: Deep Neural Network
Defines the **structure** of your neural network ( building the blueprint of NN )
Train a deep neural network to capture non-linear patterns:

It's a DNN because we have more than one hidden layer

**Architecture:**
- Input Layer: 100 features
- Hidden Layer 1: 512 neurons (100 → 512) + ReLU + Dropout(0.2)
- Hidden Layer 2: 256 neurons (512 → 256) + ReLU + Dropout(0.2)
- Hidden Layer 3: 128 neurons (256 → 128) + ReLU + Dropout(0.2)
- Hidden Layer 4: 64 neurons (128 → 64) + ReLU + Dropout(0.2)
- Hidden Layer 5: 32 neurons (64 → 32) + ReLU + Dropout(0.2)
- Output Layer: 1 neuron (32 → 1) + Sigmoid

*NOTE:* *Each of the 512 neurons is connected to ALL 100 input features (that's why it's called "fully connected" or "dense").*

```python
self.relu = nn.ReLU() --> Adds non-linearity after each layer.
# Examples:
ReLU(5) = 5     ← Positive stays positive
ReLU(-3) = 0    ← Negative becomes 0
ReLU(0) = 0
```

**Why needed?** Without ReLU, your network would just be a fancy linear regression! ReLU allows the network to learn complex, non-linear patterns.

---
```python
self.dropout = nn.Dropout(p=0.2) # - Regularization
During training, randomly "turns off" 20% of neurons. TO Prevents overfitting (memorizing training data).
```
**Analogy:**
Imagine studying for an exam:
- **Without dropout**: You memorize the exact practice problems
- **With dropout**: You practice with random questions turned off, forcing you to learn general concepts
**Visual:**
```
Normal: [●][●][●][●][●]  All neurons active

With dropout (20%):
[●][○][●][●][○]  20% randomly turned off
 ↑  ↑      ↑
Active Off  Off
```
During **testing**, all neurons are active again.

---
```python
self.sigmoid = nn.Sigmoid()
Squashes output to be between 0 and 1 (for probability).
Sigmoid(x) = 1 / (1 + e^(-x))
Sigmoid(5) = 0.993   ← Large positive → close to 1 --> Class 1
Sigmoid(-5) = 0.007  ← Large negative → close to 0 --> Class 0 # .7% probability of Class 1
```
- **The Forward Pass (`forward`)** : This defines **how data flows through the network** when making a prediction.
- **Pattern:**  --> For each hidden layer: **Linear → ReLU → Dropout** & For output layer: **Linear → Sigmoid**
```
 PyTorch tensors (PyTorch's data format)
`reshape(-1, 1)` means: "Make it a column vector"
Before: [0, 1, 1, 0, 1]
After:  [[0],
         [1],
         [1],
         [0],
         [1]]
```
---
**2. Loop through batches:**
```python
for _, (X_batch, y_batch) in enumerate(loader):
Instead of using ALL data at once, use small batches (e.g., 64 samples at a time).
```
**Why batches?**
- Faster training
- Less memory usage
- Better generalization
  
**Create DataLoader**
- Combines features (X) and labels (y)
- Splits data into batches
- Shuffles data each epoch

**Example with batch_size=64:**
```
EPOCH 1:
├─ Batch 1:  Samples 1-64     (64 samples)  → Update weights
├─ Batch 2:  Samples 65-128   (64 samples)  → Update weights
├─ ...
└─ Batch 16: Samples 961-1000 (40 samples)  → Update weights
✅ ALL 1000 samples seen in Epoch 1!
✅ Weights updated 16 times in Epoch 1!

EPOCH 2:
├─ Batch 1:  Samples 1-64     (SAME samples, but shuffled!)  → Update weights
├─ Batch 3:  Samples 129-192  → Update weights
...
└─ Batch 16: Samples 961-1000 → Update weights
✅ ALL 1000 samples seen AGAIN in Epoch 2!
✅ Weights updated 16 more times!
KEY POINT: Epoch 2 sees the SAME 1000 samples as Epoch 1, just in a different order (because of shuffling)!
```
**Why Use Batches?**
Option 1: Train on ALL data at once (batch_size=1000)
```
Epoch 1:
  - Load all 1000 samples into memory
  - Calculate predictions for all 1000
  - Calculate gradients for all 1000
  - Update weights ONCE

Problems:
❌ Needs LOTS of memory (all 1000 samples at once)
❌ Slow computation (processing 1000 samples together)
❌ Only 1 weight update per epoch (slow learning)
```
Option 2: Train on ONE sample at a time (batch_size=1) # 1000 tiny batches, each with 1 sample
```python
Epoch 1:
  - Process sample 1 → Update weights
  ...
  - Process sample 1000 → Update weights

Problems:
❌ Too many weight updates (1000 per epoch!)
❌ Very noisy gradients (each sample gives different direction)
❌ Inefficient (can't use GPU parallelization well)
```
Option 3: Train on SMALL BATCHES (batch_size=64) ✅
```
# 16 batches, each with 64 samples
Epoch 1:
  - Process batch 1 (64 samples) → Update weights
  ...
  - Process batch 16 (40 samples) → Update weights

Benefits:
✅ Moderate memory usage (only 64 samples at a time)
✅ Fast computation (GPU can process 64 in parallel)
✅ Good balance of weight updates (16 per epoch)
✅ Smoother gradients (average of 64 samples)
```
Let's say you have:
- **1000 samples** - **batch_size = 64** - **epochs = 100**
```
EPOCH 1:
  Batch 1 (samples 1-64) → Forward → Loss → Backward → Update weights
  ...
  Batch 16 (samples 961-1000) → Forward → Loss → Backward → Update weights
  ✅ Saw ALL 1000 samples once
  ✅ Updated weights 16 times

EPOCH 2:
  Shuffle data!
  Batch 1 (64 shuffled samples) → Forward → Loss → Backward → Update weights
  ...
  Batch 16 (40 shuffled samples) → Forward → Loss → Backward → Update weights
  ✅ Saw ALL 1000 samples again
  ✅ Updated weights 16 more times

TOTAL:
✅ Saw each of the 1000 samples 100 times
✅ Total weight updates: 100 epochs × 16 batches = 1,600 updates
```
---
1. **Linear layers**: Connect neurons (like roads between cities)
2. **ReLU**: Adds non-linearity (allows complex patterns)
3. **Dropout**: Prevents overfitting (regularization)
4. **Sigmoid**: Output probability (0 to 1)
5. **Epochs**: How many times to see all data
6. **Batches**: Process data in small chunks
7. **Optimizer**: Updates weights to reduce error

We build Architecture (Layers) then connect them with each other in Forward Pass

In [ ]:
class DeepNN(nn.Module): # DeepNN will iniherate of nn.Module (carry entire Pytorch's NNs)
    """
    Deep Neural Network for binary classification.
    """
    def __init__(self, input_dim=100):  # Instructor will be initialised, when we call instance of this class
        super(DeepNN, self).__init__()  # Call nn.Module constructor
        self.layer_1 = nn.Linear(input_dim, 512) # each feature connected with ALL 512 nodes
        self.layer_2 = nn.Linear(512, 256)
        self.layer_3 = nn.Linear(256, 128)
        self.layer_4 = nn.Linear(128, 64)
        self.layer_5 = nn.Linear(64, 32)
        self.layer_out = nn.Linear(32, 1)

        self.relu = nn.ReLU()  # Non-Linear
        self.dropout = nn.Dropout(p=0.4) # Probability 20 %
        self.sigmoid = nn.Sigmoid()   # [Binary classification: 0 or 1]

    def forward(self, inputs): # Connect the layers  -- Self refer to the current object
        x = self.relu(self.layer_1(inputs))
        x = self.dropout(x)
        x = self.relu(self.layer_2(x))
        x = self.dropout(x)
        x = self.relu(self.layer_3(x))
        x = self.dropout(x)
        x = self.relu(self.layer_4(x))
        x = self.dropout(x)
        x = self.relu(self.layer_5(x))
        x = self.dropout(x)
        x = self.sigmoid(self.layer_out(x))
        return x

def train_dnn_epoch(model, loader, optimizer, criterion):
    """Train the DNN for one epoch."""
    model.train()
    for _, (X_batch, y_batch) in enumerate(loader): # Instead of using ALL data at once, use small batches (e.g., 64 samples at a time).
        y_pred = model(X_batch)      # 1. Forward pass: make predictions
        loss = criterion(y_pred, y_batch)   # 2. Calculate error, Calculate how wrong the predictions are
        optimizer.zero_grad()   # 3. Clear old gradients, by default. You need to clear them each batch!
        loss.backward()    # 4. Calculate new gradients (how to adjust weights)
        optimizer.step()     # 5. Update weights (go in Forward Pass to update them)

def train_pytorch_model(model, X_train, y_train, learning_rate=LEARNING_RATE, epochs=EPOCHS, batch_size=BATCH_SIZE):
    """
    Train PyTorch model.
    """
    # Convert to PyTorch tensors
    X_tensor = torch.FloatTensor(X_train)
    y_tensor = torch.FloatTensor(y_train).reshape(-1, 1)

    # Create DataLoader
    dataset = TensorDataset(X_tensor, y_tensor)
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True)

    # Loss and optimizer
    criterion = nn.BCELoss()  # (Loss function) Measures how wrong your predictions are.
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)  # Algorithm to update weights. Adam is like "smart gradient descent" - it adapts the learning rate automatically.

    # Training loop
    print(f"Training for {epochs} epochs...")
    for epoch in range(epochs):
        train_dnn_epoch(model, dataloader, optimizer, criterion)
        if (epoch + 1) % 20 == 0:
            print(f"  Epoch {epoch+1}/{epochs} completed")

    return model

Memory Storage Check
| Concept | Your Case |
| --- | --- |
| **1 float32 value** | 4 bytes = 32 bits ✅ |
| **Your X_train** | 38,400 × 100 × 4 bytes = ~15 MB |
| **Your RAM** | 16 GB = 16,384 MB |
| **RAM usage** | ~0.09% (tiny!) |
| **Can you fit it?** | YES! Easily ✅ |

In [ ]:
def cross_validate_dnn(X_train, y_train, learning_rates, n_folds=5):
    """
    Perform k-fold cross-validation for DNN with different learning rates.
    """
    print(f"\n{n_folds}-Fold Cross-Validation for Learning Rate")
    print(f"Testing learning rates: {learning_rates}")

    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    results = {lr: [] for lr in learning_rates}

    for lr in learning_rates:
        print(f"\n  Testing LR = {lr}...")

        for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
            X_tr, X_val = X_train[train_idx], X_train[val_idx]
            y_tr, y_val = y_train[train_idx], y_train[val_idx]

            model = DeepNN(input_dim=X_train.shape[1])
            model = train_pytorch_model(model, X_tr, y_tr, learning_rate=lr, epochs=EPOCHS, batch_size=BATCH_SIZE)

            auc = evaluate_model_cv(model, X_val, y_val, model_type='pytorch')
            results[lr].append(auc)
            print(f"    Fold {fold+1}: AUC = {auc:.4f}")

        avg_auc = np.mean(results[lr])
        std_auc = np.std(results[lr])
        print(f"  LR={lr}: Mean AUC-ROC = {avg_auc:.4f} +/- {std_auc:.4f}")

    best_lr = max(learning_rates, key=lambda lr: np.mean(results[lr]))
    best_auc = np.mean(results[best_lr])

    print(f"\nBest learning rate: {best_lr} (Mean AUC-ROC = {best_auc:.4f})")

    return best_lr, results

print("\n" + "="*50)
print("EXPERIMENT 4: Deep Neural Network")
print("="*50)

print(f"\nArchitecture: Input(100) -> 512 -> 256 -> 128 -> 64 -> 32 -> Output(1)")
print(f"Activation: ReLU")
print(f"Regularization: Dropout(0.2)")

# Cross-validation to find best learning rate
learning_rates_to_test = [0.01, 0.001, 0.003, 0.008, 0.0001]
best_lr, cv_results_deep = cross_validate_dnn(
    X_train_scaled, y_train, learning_rates_to_test, n_folds=5
)

# Train final model with best learning rate on all training data
print(f"\nTraining final model with LR={best_lr}...")
deep_nn = DeepNN(input_dim=X_train_scaled.shape[1])
deep_nn = train_pytorch_model(deep_nn, X_train_scaled, y_train, learning_rate=best_lr, epochs=EPOCHS)

dnn_train_auc = evaluate_model_cv(deep_nn, X_train_scaled, y_train, model_type='pytorch')
dnn_cv_mean = np.mean(cv_results_deep[best_lr])
dnn_cv_std = np.std(cv_results_deep[best_lr])

print(f"\n{'='*50}")
print(f"Deep Neural Network Results:")
print(f"  CV AUC-ROC: {dnn_cv_mean:.4f} +/- {dnn_cv_std:.4f}")
print(f"{'='*50}")


EXPERIMENT 4: Deep Neural Network

Architecture: Input(100) -> 512 -> 256 -> 128 -> 64 -> 32 -> Output(1)
Activation: ReLU
Regularization: Dropout(0.2)

5-Fold Cross-Validation for Learning Rate
Testing learning rates: [0.01, 0.001, 0.003, 0.008, 0.0001]

  Testing LR = 0.01...
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 1: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 2: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 3: AUC = 0.5000
Training for 100 epochs...
  Epoch 20/100 completed
  Epoch 40/100 completed
  Epoch 60/100 completed
  Epoch 80/100 completed
  Epoch 100/100 completed
    Fold 4: AUC = 0.5000
Training

```python
# Find best learning rate
best_lr = max(learning_rates, key=lambda lr: np.mean(results[lr]))
    best_auc = np.mean(results[best_lr])
# or
best_auc = -np.inf
for lr in learning_rates:
    mean_auc = np.mean(results[lr])
    if mean_auc > best_auc:
        best_auc = mean_auc
        best_Ir = Ir
best_auc = np.mean(results[best_lr])
print(f"\nBest learning rate: {best_lr} (AUC-ROC = {best_auc :. 4f})")
```

## 13. Final Results Summary

Compare models based on cross-validation performance.

In [ ]:
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)

results_df = pd.DataFrame({
    'Model': [
        'Logistic Regression (L2)',
        'PCA + Logistic Regression',
        'L1 Regularization',
        'Deep Neural Network'
    ],
    'CV AUC': [
        lr_cv_mean,
        lr_pca_cv_mean,
        lr_l1_cv_mean,
        dnn_cv_mean
    ],
    'CV Std': [
        lr_cv_std,
        lr_pca_cv_std,
        lr_l1_cv_std,
        dnn_cv_std
    ]
})

print("\n", results_df.to_string(index=False))
print("\n" + "="*60)
print(f"Best Model (by CV): {results_df.loc[results_df['CV AUC'].idxmax(), 'Model']}")
print(f"Best CV AUC: {results_df['CV AUC'].max():.4f}")
print("="*60)



FINAL RESULTS SUMMARY

                     Model   CV AUC   CV Std
 Logistic Regression (L2) 0.510699 0.005421
PCA + Logistic Regression 0.502334 0.006438
        L1 Regularization 0.511300 0.005287
      Deep Neural Network 0.774852 0.006545

Best Model (by CV): Deep Neural Network
Best CV AUC: 0.7749


## 14. Generate Submission File

Create a submission CSV with predictions from your best model.



## **IMPORTANT**: This is the file you submit to Kaggle .. so follow the same formatting!

In [ ]:
def generate_submission(model, X_test, model_type='pytorch', filename='submission.csv'):
    """
    Generate submission file for Kaggle competition.

    Args:
        model: Trained model
        X_test: Test features (scaled)
        model_type: 'sklearn' or 'pytorch'
        filename: Output filename

    Returns:
        submission_df: DataFrame with predictions
    """
    print("\n" + "="*50)
    print("GENERATING SUBMISSION FILE")
    print("="*50)

    if model_type == 'sklearn':
        predictions = model.predict_proba(X_test)[:, 1]
    else:
        model.eval()
        with torch.no_grad():
            X_tensor = torch.FloatTensor(X_test)
            predictions = model(X_tensor).numpy().flatten()

    # IMPORTANT: Kaggle format: id, target
    submission_df = pd.DataFrame({
        'id': range(len(predictions)),
        'target': predictions
    })

    submission_df.to_csv(filename, index=False)

    print(f"\nSubmission file saved: {filename}")
    print(f"Total predictions: {len(predictions)}")
    print(f"\nPrediction statistics:")
    print(f"  Min: {predictions.min():.4f}")
    print(f"  Max: {predictions.max():.4f}")
    print(f"  Mean: {predictions.mean():.4f}")
    print(f"\nFirst 10 predictions:")
    print(submission_df.head(10).to_string(index=False))

    return submission_df

# Generate submission with Deep NN (best baseline model)
print("\nUsing Deep Neural Network for submission...")
submission = generate_submission(deep_nn, X_test_scaled, model_type='pytorch', filename='submission.csv')

print("\n" + "="*50)
print("SUBMISSION READY!")
print("="*50)
print("\nUpload 'submission.csv' to Kaggle to see your score!")
print("Good luck reaching >0.98 AUC-ROC!")


Using Deep Neural Network for submission...

GENERATING SUBMISSION FILE

Submission file saved: submission.csv
Total predictions: 9600

Prediction statistics:
  Min: 0.0000
  Max: 1.0000
  Mean: 0.4905

First 10 predictions:
 id   target
  0 0.512905
  1 0.591109
  2 0.716103
  3 0.211567
  4 0.020064
  5 0.091577
  6 0.676188
  7 0.963377
  8 0.423924
  9 0.116688

SUBMISSION READY!

Upload 'submission.csv' to Kaggle to see your score!
Good luck reaching >0.98 AUC-ROC!
